# Does TF32 help OpenCV DNN on the NVIDIA L4?

This experiment measures ResNet-18 and ViT-B/16 on two photographs. It compares identical FP32 ONNX files with TF32 disabled and permitted in OpenCV 5.1.0 and ONNX Runtime CUDA, then checks every output logit against an OpenCV CPU reference. Allowing TF32 gives the libraries a choice of math kernels.

In Colab, choose **Runtime → Change runtime type → L4 GPU**, then **Run all**. The notebook contains the source and downloads pinned public dependencies, model weights and sample photographs. Build files stay on the local VM. At the end, the browser downloads a ZIP with timing samples, placement evidence, predictions, exported-file hashes, environment details and the code. After the download completes, use **Runtime → Disconnect and delete runtime** to release the L4.

In [ ]:
import hashlib
import json
import math
import os
import select
import shutil
import signal
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/content/opencv51-l4')
SRC, ASSETS, BUILD = ROOT / 'src', ROOT / 'assets', ROOT / 'cuda-build'
STARTED = time.monotonic()
DEADLINE = STARTED + 110 * 60  # Leave ten minutes in a two-hour session for retrieval.
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = ROOT / 'results' / STAMP
for folder in (SRC, ASSETS, OUT):
    folder.mkdir(parents=True, exist_ok=True)

def run(cmd, log_name, env=None):
    """Log the full command output and stream the last status line every 15 seconds."""
    log = OUT / (log_name + '.log')
    if time.monotonic() >= DEADLINE:
        raise TimeoutError('Experiment budget reached. Completed cases are in ' + str(OUT))
    with log.open('w') as stream:
        p = subprocess.Popen(cmd, cwd=SRC, stdout=stream, stderr=subprocess.STDOUT,
                             env=env, start_new_session=True)
        try:
            while p.poll() is None:
                if time.monotonic() >= DEADLINE:
                    raise TimeoutError('Experiment budget reached. Retrieve partial results from ' + str(OUT))
                try:
                    p.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    print(log_name, '—', log.read_text(errors='replace').splitlines()[-1:],
                          flush=True)
            if p.returncode:
                print('\n'.join(log.read_text(errors='replace').splitlines()[-50:]))
                raise RuntimeError(f'{log_name} exited with {p.returncode}: {log}')
        finally:
            if p.poll() is None:
                os.killpg(p.pid, signal.SIGTERM)
                try:
                    p.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    os.killpg(p.pid, signal.SIGKILL)
                    p.wait()

def output(cmd):
    return subprocess.check_output(cmd, text=True).strip()

gpu = output(['nvidia-smi', '--query-gpu=name,compute_cap,driver_version,memory.total,uuid',
              '--format=csv,noheader'])
print(gpu)
name, cc, driver, vram, uuid = [f.strip() for f in gpu.split(',')]
if len(gpu.splitlines()) != 1 or name != 'NVIDIA L4' or float(cc) < 8:
    raise RuntimeError('Select one NVIDIA L4 in Runtime → Change runtime type, then Run all.')
if shutil.disk_usage(ROOT).free < 20 * 1024**3:
    raise RuntimeError('At least 20 GiB free local storage is required for this source build.')

In [ ]:
# @title Unpack the companion source and verify every file
# Expand this cell to see its source manifest. The decoded scripts are also in Colab's Files pane.
import base64, gzip
bundle = {
  "build_cuda.sh": [
    "590acc2609d36c5980807b4a67ecd55f82ff556ba31ee47d6b5e87795945b292",
    "H4sIAAAAAAAC/4VXW3PiuBJ+51f0eLMD1GJzS8hl1lvlgJOwQ8AFTjapmRyXsAV4Y2SPZTOhMjm//bRkc3EudfIQrFar1devW799qqc8rk99VqdsBVPCF6Xf4Dz1Aw8I88BnPCFBAMmCQvIzxPWMxpS5FCib+4xypCQhsoLPw4Ak1IOBz9InWKE4rcRpAipNQ4j8iM6IH5SmQrQTh2GiKwfP5zf9Qc8Zj0b2mXpg/dOra2GE0ldHTdVNPfKilJaPnh+DGoFysDuqFMVUXK+4DZ8/Q/TTqyolIWV7W/emZ+SXSauD0CVBPb+IPkVhnIBl2FfIuz0nXHN2IKgKOmZCA+om0huX1k2ZAyOJv6LSTXHqJn7IAG2uAdXmGnSaMAtjIHBJL8IYfXZp30GzcdIA29cy1UjsLraqGePuFfqhwla+5xOVL31Q1R8pjdfqPEp1N1xGaUIdl0RIR8lLkuguX9VYuKDEozH8AvLzEcrDMeg6NOF5ztNppf5Ne6jXQFGqXyCKMVwv5Sra++0b5GYKHRTQ/wv/+dZQTx/+OICHB/j1C56BuosQyhMM4lY/wGgL81m6pLHvQq4UoFJk6gd+soaKNP6kU9XK8Nfn1hegT34CzS/wUmIr10XdVzTm6KoSetSAGH1KOKYXmWMSoUCGueNTlkjn8QWmX/gT3CDEbNPgFm+drcFPMPOWyzQh04AKJZY+ej0JQ/Sr4HRyqZUqPJcA/2Ss8a4oRHc3FSEXP1qK3PRngO74BGoxj+oHgr+uzX1MqYeHL8JwJg+IP6RmWqFBHo2SBXpcVacxYe4CjrSm1gBlkSQRP6vXkXeRTjXUs54leCY6k/z2ykyrmb+nHGa5uFDtvseOdq3UiMToxSvT6FUV+KQjH9qoiECq7P8cxxpPUo7aRyJNA+Iz/E5ZEhP3kXrqzA8o11lYfccLMkUU8ynCusDiJ+gSimgwQiu7t7kb3AV1H8M0wTKBd27HHNmJk6myMf+lGEzIfAf0cHZ8Mpu6syY5Pm7PptTrnNLm4VGjQU+mx+3D006bTo9Ojt497WCCJbE/hWnnmJwcHTVPTkjb65y0T5rt43az0WzMOl77lLid5uFxu00OS6VonSxC1gZ1KWGt6EJBUXIW/c2ORNZsUykpB/mXkISIuEXXMlZTtNb1ltbSOmUoh4w9xeh/f0ll5etNrdXSGvmOWDZPcInl46bnA2MiCgCheBaHS1mcuVyMB1ZE8OgnGljoe7Hlpr3hEHLhEuIz7OAoLGeRNvIQRmNbMuSxTEWN4jYneNAiHOtJFdWPCChqUBqufWyjqrJQ1AlHazN8Q/BhDP83W7p+qjUbaHyrWS5xP6EC1PckuVD2lxKe+Zpj/Gb+PMeyypagzWnioDKLihKlGHJ/qlSr5awFMBEZIbeeXV2XNAHomW0I5IHPHhFBqUxkLg1NWY5T6EePrmgQRkuEJRVFxyReI/YvqVYKsFb4jKGAqZSq8VA7zaCVsTpS63s74koD28QKncbmeAsCnqguWJJHDKAMDirjrVG2jw4O1ghcGB4PUDlGAw5khV1Ugl4SorBpiLiza8WYApQvENxoECBYfqU0ykzhNEa5WF1cXrwxQfgLbSc80UrSnzMo8zAVver3H99Z3hOxRQ/652NjfO/I9vj7j4PnV7SzP84OXpFevrMyfJel/H5ZkMwRVNn31tle68Vl5xDxoXh+c0zjC6WUK/sxg6wsJ8/xD9NKcm1SSi5kOvnMDVIPm4hMJFFtmAfA0ygKfJolCRJ8VCC/AIH4R+rHGK/pelM3bhhTDaci7FDyhNQ0i55YWlIbQId4GJkadjUyp9Cvj2qoDo3i0KWci5iJUsT0QHKQyrtxLsJ6osxDWEN1sPOJLAJ1UnRHBnto7nmRLj+RfAlDn/1L8lCpve618dV0ssnMvrdMfZzjZ5GhP5zYxmDgWGPzon9XRL685pUPjozzIevtCRHyL7tk2BMg5g97NBp87dtyhHN6/fH+nPaKVYwqznl/qBdmnP1Ny0ald6dGljns3jrmnT02nOtR72ZgTpy3ehZ7SH0Zein2xsLlw+GmDPSPUGDL2B92Bzc9c2cMMudJt5OZhWLQn9i6TMCayKiav5yL5BC/buhRl9fmFLtAEq9rKKWW96z818nTiztzymhMEpysNuL/6dtXjvCLPhrulqKtFAiobrbOPSXU35wqamqbExvPXlxsKZY5vniHbN4Z1xb6WRKLMv42bo0Cb+54EkU8p0u9pDKDggBJ71vWazbzbrxPuri4tszLtycvJ/bYNK7NAvOl/XV/eZsvt1G3bpxzY2IO+kNTn0zMdk7r9SeYQt0r3bi9a225rXv7ajRso/lm98Y2zgemvgdK2+3hzbV1v58hE2QrwJnyRqRldL8al7vMFT1P2cVsw7at3T2uDXqoGT69CxVixhRzRQDbt9vfo/PJmdp52Z3fNPx3JIiBv5cBJpUPCCJfTALZpmLUxN+YzrFHbTZiKkBQgqpQEluaLRAUU2GxJPGjgGQxW2JnIzh0YzXSANtcFBCXilaNbzGhcUKxj06pS9JsCszyX8C4wP4ViSV67sUA/vyzbN2XS3l3cJN1RHlpu1y1Np97gxoQDkgSJmbDjHxsxDhLnDYbjRYCdhym84XE/CidBuLtJGexfMgAw+prJcKxU29u1Lq9waCiFCYLparJxSVNbrOD+MTB1568ZHt81dIcJxfsOGJfkaO4Unu9t39EIKXoe138NZkYMrwsWt0QbcR7/oIGPiSHoXwOblrfCkOTzSN5x1P2ZQq9BVTYxvjStCViiDlls4W3GZuJxiYxLnll/9g5JrQ57Mlz1Y1cRazMJ+qmIkusOMShjsaKkIsOl517OyY5Ub7NK9VsxKnMlLwzP7/yxUsNRsPhHYzziD4LacX9YrfeSBAjHG4r1ZJ1X8peRPmUl43MOPH4cchkSv7Ep+AZ5HPL948Gl++KUvof/WTan5wRAAA="
  ],
  "benchmark.py": [
    "e65f956f473c8f0ba80dd1d341b45cf3adc9dfa1f7682485708b9d310b790a7f",
    "H4sIAAAAAAAC/5VabXPcNpL+rl+BU+qKpDNDW0qc253s7JUjyy7VeiWdZXuvSp5lcUiMhhaH5BKkpIlWX+8H3E+8X3JPNwASnBcl60rsIQg0Gv36dIOHh4d/lbFqaynKQoqzVXwjz2UjkjxWKltksn45j5NbWaQiK0QsFrVUS1HVZSKVEjGG72SdLdYia5So8jiRK1k04eHh4cFBtqrKuhFxfVPFtZIHi7pciaTMc5k0WVkoYSaclG3RyNrOX8ZqmWdz+/hNlYVeuoqbpV2TJys7oVT6dYXXWGdnXOLRTqll/0uVbZ10z6qdm8N0I2tDr8lW0hKrZL2IEs1nVKiDg4NULgQfK8L5lB9MDgT+VGLanTd8U9+0JI1Leqr9VKqkzio6+jSK0jKJokAvCuM0JTI82/fG41WZytwbiWZdySkdZATG/9FmtUynn+pWjsRS5tXUU1lxk8txVlRtMxLmqWwbPIqL8/P/Fossl96+XTJS9nCXmDUz9eKqgsq9PdvWspJxIxZlLVZt3mRVToICMbV3rzyey1wNN9PE2NJEEa+kGrENQtYiz/ADBmfOUtaprPfSNgYK4smyzKDLqe+V4D+5GydVi+HuqU1jfqwb/TvYOOC+HTQbv0MhhbwHs0KuqmZNtgbpiBSzkqas188coEmWkqQDq4qxZuodeZYkv4Ruf5WKJELbSmVEMhIyvAnF0eh4L+1m8cPxUDIFi2CxoNN3+9Gz3THO8/JefHr3wzH0IU4+v32zn/qylnHa6TUrmp7mT5beBaR/8kWYuRw0Lj5+wmGaOh6XlX2xdxPIEacfqyX0tnOn49fdXjxJyPRGirmEgUqRSHJakdS0UymOj3/cu9F9XK/aaucWR6/sDmmmkrhOwXWmEJKKNfkBVuJkZkcVryoY8M3+A7XFbpG9ftW7GKuZVS5jGMBKR+lU5GVZ7T9BWd/CU1x9UxgjjSPMlaxz3qALUlefLy8/nl5daYoUyxDCqtANbfymqdc6xNlpobFaTL/GIXwVcEBgjt33oYIwwNzIC2ZMQD4ksmrElzhv5Wldl3VPtwolDTg+gQADY59DjeVqFY+VBGNxAzFUpcqa7I6iRCNvZG3NJ1uIVVb4LgeB+JM4IrfsXpACRppLrXPzYCyRF+zgynFF+D/T6JYXaWfgzDKciNi2bPbcGc50Tp1OhRuqmIzmBG6r3xbeDk7YOWF/MbunULJpeoOzm2i/ibRL/Iksf6ekXffqxI3wnsPiGsdfSLsVp+BCXPMGnKZG4gU/cAKYie+Fr1/qiD/ruNHPAv9LcT0LelYwoSgbJh1mKqKU5TuvXW4X3ju85ekLJON0Ih5p2ZPhsJZNW2vrMxk6iWHFNEWx4g3Z78TJOgEdKKxsb2xwJdnzIXRsxTFTSVkQrpWvfxbNfSl+mR6JzjOTuDDUEA/qNeL8YiFrzBZV9oBjhuIj50koBjtJ5DSIMymBlxTnuUULE7HhPCGGQqZHPLNXzTqxZxz54+JG+gA+fi6NHTO7cOluwFp84AhQW+104JTXmfj37VWzbg3zEGoUYPTJe137GTT8LXBXayZmzOi3nlEmGswGmmGyRjU3VRut5Ap50e/U8l+thCBJWCaQQVL3lPOaGIAgFe8vPwu9RtxnsEQtVgQBkFUALkI1GYRKHhUakheYU4tWsdQXvIKopCVbEUO6ULzBj4L3QJCI16KQ2Gw8rrJ0uixVY/SySkkrXnGXpRlC0SqjsDoe/4OYHiM+ASLIMWSmplg4ovO1LX5g7zTqT+uNXNPGesgNyHaaqLtRUS4RQZDXC/AFRO3NtmNvXd6TdfTAlaKZD+Yo6Mvk1uCRJK4gchlp6GIGG/mgfwahalK8cgOyf3HF4Xjk0j5BHJPppX7i145lGa2ew5pH/HdnsBa/EbM6AdCIcv0atUWesp3fUSYAQwDHvk4iPELradVWAjFBgwxQEwkoUv7ADqwHrl/NaAgk/VKFN7KBQvwgcGcczxBs0uwGlDeCjTkV5bRuLnzM/D6aHew+urZq1SJNwaQZA1CasL6l9zDlRNGuqrWIAXsrl9qjV71+Fa2UNxGLvIwbv6jClYSxFZZeEFAm/+PrjUmAzARzKHJ2G//xNSYPDuaZd3pxN9Gz8HNimX0yh7mvs0ZGVID5FWPeNG5iW+lQuNYTyKp8mhWmOJfyadaIw2cBbDYSjCejIi6m72JE/wARxPtKKBR5r0wRHqde2yzGf0AUN/uy8w8j9hlLThkVPpBtowzVcIA8H5kom0vCBvmaXiQyleztKDjTsY0PBIbq0FVFcndMikjuntOPrly7krhbS/4WqQSgD3JcxbcymuflfCR0So/gmAWBdMztfjdlhRR3B3sxZpzchVFESQH8RZE19rEXkBH/GxDA6/AofOWAgDrOkEI/InDhMKcmL34iNDrHrkuY361oIDsJaRnozSR+NmwjtD0ON7X5E6NAEuft6pNGMv4AFPEUOva09/WdeMZWV06KN5V6UTzUmm8SLoa0FAw0mLpAIaRNtWltmckgouwBGINUugkD+CWpiuNPpzi2cgsMXfQUdOCH4S2TNaH5FcHlYXfA72AYljoBT5cJoy5ZTR3DYL7YUkbUXSEQNNX4lH2BBWsQvI7ZOOUWaaLoWp5z2OZoL5/afvHaUPFZMLA+s/J470rX/n3rBhSWWUBEYWR0GxwMzGXrUP1RvhOnD4hmZOLyjtCAWsaVfMktDioC5lmhHZ7KjEJoa0oB4eZUctnOVOiKhjVN8/lgw2Dvnjlws8tAdfSHqn7fqC6ErUdaTdgUmQXWws0v7TGma/ESp7MdsbFRakhx0gs2UqiWDkUqyPrRQ1hPdZD2obcxjCwAsSOJKtdbZLVqIsO2nXRMk47spGHQZ2JNGemFuicxWGeJP+2Cm06NFGzZm7EUiyVZ64T4TMj+RH7OxatBuZQUkrwlXxaEq16m8g5VKg5eZQaA0/D4LiN1StP9sUjuBBAtK1od1ZG9pAbNIkZprxD8SgBCKo5hg7Ccgk9SpimXaOO20ljPWEa1BK824OjjaV9o4oZc4bEXxkoCmCRc9Hjm3IT7yAIgVeo7zXrBbKZ9hvEjG472yZdKyQ0QcouVvubye5EFAwkPJoJjDsm7PdQxaHMO8jl7jOGWz8SC29mQ3DD69FEehUBky40+qupocDvbFVh/mwOH6vZsm0376GPCzYCkVootZ/wtUcELWIzGEYarbRvDrO7DG4TSz2TDudYynjE834MD+wpcK9fYIJmA9Zur+M5Uh3kJfEppoW1UZvAMQ5gR2xEbvm4Wj+WDbrbDh2yK5fBsDb4GjtRtd1JDHa+tbTqmfjvqc5wEBmI4ZYyuV/etXJN6PE6qj7dP3sG/YD17UwZvHGwrUPN6jU1ZrLwiRMBYOxaepSDCHWRM+U2TcERhlfrogT4iIv6Ga5tW9kRcM8glwoPmB7M6G20Z4yDssr5Bwz2bB95eR+AW48wzDRxFpjc+0Ud40pxywZgBS1INiWO55fIm7KJ0pKgg9j3d1tYgyO+rk8CY1gcE0AeTdJT4S/YLwdbk4krM1w3kKhhGLrOb5fg+brgYBpg0PYu+LB9m2VqR2O3FClVbdasgQb8b+vj56s370+jq9MM7ZEqcI37AIruYjkkEQOal8I9e/fiH1//xEx1RrVWIBNpQgcxxK43r+6zw9OmOXh3/qI+FAkjBZX7dTL7ae8Ki+hUB+8ULbUl6CZUonGrt3cFkIFBoRl/BmGF+YEMgnVFHfdK36TbybV9Nub0VWmfa3JNBr3FjNbeGJ8LpUnqmKz0Z9is9QgsYpH/odByANohpMEiTBnCf2v89EsdrBA73vYE/WsxsQ0Oypo8xoXpVxreR0SEGzC9sQOYaLyjAEsPmdWfTnm2OYJD+edrYwXFRzHCedKxy6tINleNAABcaZJly1d7VWSQGCESR0u9UDTPpMDcXCbZWX4hdbdphZfOddpmz83cX3F2hpixXi+eymUxQtOaTyQKQNUfKe1/HFQcQ8qQqK6ijxZUZnIearTLcbPLUMsTiFAW0Xx9aMsD44jv/a/p9IDw/fPGfgSe++v7134PZ98FX5LI/C//k8vM/qTEcHPLxAofdNxpQIxAT/uBuV0e4A1Y3zGl8E2dFqK9nNY4jD0hFAW9wKFLK6aSLGi+Bk3zjFriOGXTXMwfawfpUpi1KtwSxRVmRM1hrHMQOBEepyb/NSDsaIuoCDJaL/9xxkqbimhbQvPFJbIHTJUL+gQJrpsctKbc3BcFsFZLkf4Vt2N4vy9zKgi7CLj97tic1aER1u4A4NRXjYu0b9shw+ALN8t+zjWNuYL6U2pw5eKXuSPitzArfkg6ItndedrpKdeMfRokcXdacsvvrdxzNG/a0dvUMTHOACfUK5AwtFuBEUn9dM/W0s+zUmgPD3PyhQkP5riM9V8O7NqktrC0yVAlCPsikbYyZKYvu9ZUzRrvLNhwYluVZOyKlQXm9GenliPG+vPbIcD2AHfykiOHNruHP0Y5RELvL6LJ5pk2OVaXPuZ3voXvOd0i7ceNxFxI6SqW+xzGvmDLBrEA/dzsET3tM3lgMqNu5nR2dsnAQCS8tFR2l7cQthGzMThPuyPVew1Ia9lX3mjAdil4SsZABswEdf5WrpFp7WmKb7rlt5zuM0aMgYQNCmul9UCHci1iYzjortLeO+dq9mU6qFgI0H5KwGPfxQifsB3yPvJpEfvl5W7ZBMOzOUuyPQIO6p+3qd/r4RmoDpw4JPIXc7Eb8IYCo30Z0M0xTKO3RHHrF0SjiaBTRXeFEcDfV9mrpEmDQMP0ov8kE8ThP7VcUqkRYixv6ioe9mw5Wr+keSxTUaxFUQ0AzUC20oNqEMB7qEPMhgwnRTsYNV7dpVgPe0q2XMncM8iFTTVTeOh9UEFyFQbkrkcBrWuteVGm76O+FEaJOliVxFIvBhxX2g4z+0woEKoe6jVZcYrFnbdyCz34ngjbr+os4XbRRrGXSPe+IuLad0KER3xtGZPZo3XjoGl96H4tbaFbvwrK4A022gVKBw7usLouRuLg8PT/5En24eB99OP1y+mHqEfp4jvrf3nw8Pzt/75A2d1mEsLVHxfMcuqMLE+5zRXwNG0XU2FJljmqKzPMFzccZ766PJjO+++q+N+Cd+zDCt3OUYAk1+d49BAo0APIyXg2DAXl3Lsmjd99p4dhT/E+80YXVVNPgR1nXU2fJ1ae3F58/DZp4HfFQ+zBy0EbfYivjEtNO2iI7VPzFE6lyo/V8PT56NdlojezMtI8snidbPhn38x938fcU/CyUlOIRnDx9LR5t+nXrFiffPoeAA+ckQeeK1jZ2WDGRv+6t1qOSews4Dwul35DX81Xy1p9/pX06xCK7WH+0n9FIKjQ41JsCgszTqey0ldZzbaYLh2p2Q7fzU/tRYgjwfPz6J6f3wH2YZVvckjwpqvl5vJqn8UQsWC4+1anihS5XR2LueZuoj7cI2yrlZguR6mvUa12HRnpbPpSZv5QP+pfvznZ7azx7q+PmTOZ0ZnqtkU5aihfpO1N4nc+JLfpydnX2y4fT6O3pl7OT0yvP3VHfhkdUCEfUJ6iROTeJnH85e3v2JqKPVqKLL6cfP569PfVsS+JvVMzpkqidozawn82VRb4WXEKai4BNiMoJSqYmLTWraisAe5pUiHfeZuGIMVsgmuX7eviGiGN5z/tcWyA83Bq1VDXfIx8+ul7zRLiapWc7sYQ89dXvbBIeL57ESvX3RYeOvYCSXun2n3cspqq1KfkfHy+DQ+cbHrJU4y5One10sZ1hCgDXgzbVjPLwNWUKVFt0x8NE8ZsDynXf45rNHIIkBcq/OsgiInKT7Un83//8LyIdUX7S+InHTdOOKP6aVT6omp7cbOQyFwSukJ9V28EBIl/EiD+KOENG0QrlbRR59j67VpyEnv/yLTNf/abmDmvoy+biWs9wEvlWu5zSmzvLfoLRg5/RIIdwXIJzOTcNnLqzxidN4A1E8f91fzAczC0AAA=="
  ],
  "classifier.py": [
    "c15b88f92d7f2f7ba00a52196730da66dc8bfeef678fc004f67fd953e24814d9",
    "H4sIAAAAAAAC/51YbW/bOBL+7l8xm3ywvKeorXdT3BnnD4njZoNtbcPJtncIAoGRqFioLAkk5cSby3+/Z0jKVpK2i7sAu7bJ4XBennlm2IODg4u1uJMzaSgphNZ5lifC5FVJ97lZ0byW5eQz1UrWqkok9ss7EmVKgpJVlSeSqozms9m/SDWlyddSRwcHB71evq4rBZWbIQmNj3ahbNb1lpfKutfrTZbzBY1pOPy1d0g7O4wSecn3aANLtMkTTXlJy/NTqlQqVUS3RXX7QVVre0R/gaELocRak25ucToxGvrMStJaitKaix/4UtdFLrXd0YkoJJmKpEhW8EWUpSxIZEYq0veQZANO7dll1Ps0PZnB0OBt9Ovfj0PCx/F7+/H2/aB3eXXm9obDf/Ai3HEfxwP4mMqM1uKrjNnooBZmpUNSUud/ylivEJPx8Pj9YNQj/CF0ZzKpUmlNzPJC7mR5JYdpfIJkesc7bJySplHsI31Y/DKk2eS3L3QrTLKyeWCliapqDQOvb+zPrFLEVnBIrTXuav7LOZyQTDZRvlZSpIE2ypo8CHnx4tNyenIWT+Yf58vB/lTmD+aaZlUp9/r4D7nUkj6LopFTpSoVZH2PqaRqipTKylDqfH7km576e83w+G5lQiAxhb1jd02kV6KW16PhzU7OBWVM67wMnp3Zq7IhRJKcqp+fJYDevHEqQn/j9/b36tx26mLlfgTWutDeFCK4AFJdFbaUxhy72dV0GX+8mE1POrE7pIlkSZskW3FVY4B7ZDVZMQIZB5XK7/JSAJ26lolBSKE02ul4CGkLQ1TVlGkQ8PXXb2/oiLi4BvSGhkhed/Pd882dHouTCMiXEPUOXm9HW/qblQ7pYfTgv98A2HykdlVno5CWZWTrcXgKoLt6DAYdsajNgdNmLXi2japbnkLgSjWSQ+Mw6BGu6fR8GdFphfSBjSxHIAFrAKfQJB9sZEARLi6HNC9lS16o9AJoW9eNQfEHdf6ASofrx8cIBDMEB0Kb1NZTIbaaOAmzcBL+Fn6JuiZaOhmTaepCBnz+Z9owsm1RuW+oKiaL554x12SgJUi1p9/hzq6KwXMd4BSnwte3D/D3iC+wyQv9hS3tmKqOs3wjg6K6yw22C3GLaO3JZinKr44ALXj7INCkQuojuqwysxYPdG+LtJBi0zKQpWCGZlMya97JdMc0dsuhQaNcLtKH3c1Yupwvr+Lp5+ny3/Fy/oX+s1s7m15OprOzi9n54HoU0ujYVXaeMrIc45uqQCPwcDqkE+eJZUimnarmMkOF6IoYM1Kh5yBVRm1bhNhuJijNs0wq1BxtqkTcNoVQW5fiUqB5MU26IF2r6h51dGPTgu+cFFh0w3TnJAj/yZZVfZogETpNPgUVIJhsYjTHUqpAqLtO9CdVmeV3jXJsX6Lz3UrcBo+MhmtKG778Xqg09P0rSWRtaFVhJy8B5zcAKj52CWAdu2LkQKCdMly4QVsuZwMiGxHH6Cw3nZ2DleL5YjqbfHYRho9W8lYkX8EGNB5T33lylDSp6O8pHjdGWpqFkgiruC3kqTsSeOVns1l8ejL5HfmNJ3+cnQy+f/IKV0rTPXh1sjyfXn3jnCztgQ+/DJ1HJkPrs0aWvn1wbv4/K7tx+B/tXPzheZEz71MXcMkOXplywelze52mYjHEEu3hAaP9OHoH+qrbyQX12RTAClzg+cWiwQPHaUBxRr0uKr22FpLKdPEYku18GO+4msYfBCLXYQinge/lNNB0QZqnQMyHGOLuZcFdyQ1UQL015kgoJbbtpa4TgvzkfiABhADTPiucPsik4eJdqGqTo3D7LQCtFE8HKDxYGCHmsdiIvOAcxLUXR4vZx9bNGks3iLppo39RYopkK639Rw45qRtZvSTGM99tt1WjnJ/cCpIGyY08oEAxjo5MdOkCMLek0/Y43o/gqxIxWBdjbmxWXIF8yGHU/Wzd8/HeG28VOPNit8kWuW74XMgfje3/0Auz/AFyu/J2nID20oexR3WB0K9BeP2WPD9aMgfxOk/PlahXoNAsGyGPIFPulELlGilutMdcWak16NVizec/AoboZHHhG6TPR+yomL1+7KdygxdCnKf9EfXf9kPqQ2HMxYoFthfx+kYBD56syhZnLugXpeXtRProv6Yzjk24M0SPrwOGUPjKtMFNGwmOAbcJhflzg86VgSp34sBdbnJR5H+651AG6IGrNd1uESKgSjWJsa8hQuF7jW1s6KRAK7Klg8BaQl9sMdzhVaV4wlJotTzEbI+q8mi6OPLKKW1sa0Vog0HUQqVVyhXQwT1PeT+NbZ38sAQwcMNPvgG4B2Hs/HIlPaLH71zQjuLtdppri84M9cTs45GPuaYRRexS+E1j28jDZrb2hjeDPRb2PaejCZ69PH69P3LzFx5P8JwDdSTcYdXaJro9CwsN521Ed5B47Nz5k2o9ts2VmdGWkn7hltsNALnuqpcNdt4Usgyc6IC9eQcY2zUv6BZf+tF5KPWvbDX6RzkA6YZclCpGW/vOs9otVfGSG7W8zS1t8SzCLydrBxAT8cJfNqlD4tGScU2zZr3Y+vN+3EBgOKCgulJnPGqlwghrht5iJlRVCWx1e1X0ssV1KCTgJ2NIj2zXyD7snxjZ3+pfu3j7RpasZPI1dhNr4D5C9/Z9MelyDbm5tkzzNcd9yMnwa+496YrJnn61hxfTP3+Yqqw/tblBgV17A2zepL4JHcy6+p7sTOmuerQfT/sa8LOlfYoALN4Ptu2FST8057Fz9qnVybc+vtDy5PHS2rs3hOunrKNcZ0wY0gd4EKH4X/fcLmw/MR/71En+h6ISHOeUOB9wx38BK/PW83cSAAA="
  ],
  "test_benchmark.py": [
    "9bc0e051a7c33da65925ae93b72aecedbee6ab7b094387dd39a19c1169b485b0",
    "H4sIAAAAAAAC/7VWUU/jOBB+768YdR+SnEquBMruInHSLXB7L6wQC0/dynKTaeslsYPtFHqI/34zSQNtw+rYO12kxrU9Hn/+5ptx+v3+10LmOVicW3ROGQ3pAtNbBzNjobRYWpPyhJ6D1Bn4BcLpzdnvUOYyxQK1h7n0GPf7/V5PFaWxHr47o3szawoopV/kagrriUvqNhMei3KmcmxnrpFbaVdnymLqjV21ziqtvEfnn72nywSko6Yd0FVRrnhIl73G+xR1uiikvW3dP4MVdCAaaMzSXNLBZgpta1fIWxTT3Ex7vV49C59aV9eEwYUtmpi7p9JhdNwDejKcAY8LO58KbSxxqv6SnugUxJqYSp8uhLEZ2tBhPluv4ucdnCnnlU7pZAupNeaOvHkiAabGL4h0kGmqMgIvc/j0+QoYXx0Li0u0DjMoUGoXP7tMTW6sgxMiJJbWylU4Hh8M4Ih++wfDyQDGyTCh/x8HkCQT6pNdpbT/ED27uFe0dTcoYcQ8W2P8ywH44TjzhuPJ1jBLSA0aPKA0IIUKLcklbCBG215aT+SIlRLyPhH8CrPgUT3FpZ4HHXMmM6Y4ofXXtiK/y1gV91bRFs7bkL1F9flmVZ6HYZIcDuAgGdIrWuPaOH0UvQrHxbIsUWeNty2TOhQnL7KpTdyLzQa887tK5iEbxW4hSxxASDE44BAc1q+N3fGhJL4xYzWR+4as8fEAjo/39ifESDIadYzJMNxauAfjYXz4YTQAakZHdTM8mjCjNJEkH3mk3p2b0UvoiBDWMqX8GrugCpHmxmGNvwZC8hkOWUOsp3bbAUhv8pMR7r2Pett5YYjBdEnp950snZiRx6lMbwVJWIv7Bb2kSKtMilxpFMoJKj2ONL+bLrUwN2i9ksqhu8I5PoRXlfaqwHNrOazB/cJQgZlbWS6Aytrp5U2wI7jdshAGDcw9RhIMoD9TmhMZwZTwbh+C1OhlAOEpNUkEe7/VpfCb7nd0s/UEttKaCygXz1dQDeCL0Rj9mzN+MdBizGow/8MRL2/6LcKdWoclSlbb2kdT7+rA2SU6QUvEy/4lWqqMGe5GlAPOtaOD5O6WgVxIf1HlG2RvmSUQODL6yhnVot2MR3No8v5PPDQofoFkNxidFG7Wj4NaraZ0ASXA/lvsy4rMhV+VWK95DGrUwTHsP3WyxTLOu4rqrhOpKcrKI9FZZ8guf5xBvi6+j0EqPfkjVWRIZwq0LNg/xbRccV/auaP+45ZCiAjRGl5gkZarP+hy/NM4z0vo+l/S5WN5lvk/f8C04jhfthNPT5P/cm3sFnvekb8MYv6KCDrmcV3chccHH7JFnNHt78KGhJ0C/hN5tOYYWKDBa9dSVz7Wt9oJ6Ne9GxpI7d3xw9gU0hdV/sPodCJUp8NbIxP9HH+s/x0OO3J+GxPb6ZGQvntqBqI+hhBwcgKBEIVUWoigYfv5y4pHw6j3N7h8l6OaCgAA"
  ],
  "prepare_assets.py": [
    "76ec6160fb6d29c04d2d6ed7224c1ffa0d1bdc7f205a7eda8eec7e2647a61050",
    "H4sIAAAAAAAC/5VYa2/cNhb97l/BVb9o0DE9mvE8bGA+pInbBk0TI3HaBbyGQFHUDGuNpBU5fmzg/77nknqNnSa7BmxLInnv5X2ce8ggCC4eqrK2zN6XrKqVrYUuVMre7sRGvVeW7cpU5YaJImV3qtbZI7NbtWNig3nG0ouu2eXjVVnLLbtXerO1hgdBcHSkd06wqDeVqI1q37fCbHOdtK9/mbJon63aVZnOu6n7OsdMXqt/75WxR1ld7lglLC1nzZRLvHaqiv2uemTCsKJqP5VF8TB8rveF1TtFk/BpOGJyvesMod0cvNxpo2GoM+Hy7btWvXOT/zqYxhunNZM+KgNPRqv4T++eMftDX8U/xdGi/1IrU9CcMbvTNk4wdnR09AO72iqjGrcynSoYn2lVY4eCNsFMpSS+SObiposNwwY5e3Px86vP766YLPd5yuRWFBvFdHFg49HvH95cvPvE1uzLEcNP0JoQnLOwN+e58fzt769+uXh/cRX9Fv8Rjd3Sg58gmy0ny2mWni7V6kzO1Wq6XEwjOU9ElonlfDJZRWfZcnW2SBIhlYwWkyQS0ULOFxM1WSoZjLzUoPUEGdQ+v/Td9w2Sq8UyTc6idKaiqVzIRKQiSRbRJF3O5PRUzE8XSbaapis5mYmzTInZaXY6E8tomqbJRJ2RQU8UjZ9Ku2XVtrQlCqJW7PXrCWevtwi1EkwbljyyT1ZlomB3+E1Vzf4UuR0jDFmmFA1/FBLT2e8a/6zVHEJf1yrV1pyzrbWVOT85MVLfanusKbV4WW9O0lKakwmfzvnDiaj0ibn1Y6mwgm/tLj+6/PXD1Yf488d3iGbQyqnFPd9ou90ne6NqWRYW6cNluTvQcPhy59RMWxUnpOIk8PIHqSL9nnlVbBCcYH62ECJSS5mslnOVLM/gvqWaRZPZKkqni2Q2W4lVJKepmM5OZ2fLyamA508X80it5GmSBE3AvZ9aqVJOptlKimi1SqLFUi6Xc8RkEiXzdImAQl+0PJvKDNklZ7N0kYloLk7nZ2fJXKxStQxc1I5SlTGzFdP5IiTsGJ17XUHwydZK7DzAMYIdAzSry/1mS6jmcAoFw8RdqVPgH1MPKDLEEhBTZm7KEO5I6L2m/IASXlaqCIM6CUaENOTF8y4xAbL7umhxkJPmONUbIFxIE8cs8OYGI75VD83IqNlJWt4XeSnSENg4drrGMAwoYFXab+1DkT8yRJ1Jyre0W2XY/VYVDWgTmCCJP/366hja2E5YzO03ozO/FW1iMjEcuR4w9CRbrzvdz7fnXn9gvylVAabQAKwWOSCX7QvXRvTAKlbuLbk0ofIivzqrnTp6S1QhtztR3/LeyW2n4FeKMFbUj2+wFwl8ewxTXa+d5dCKPboIGCs2QMfeStuuQ1JTDwmbGSN24rdNANvNPmxEHK/UKbW6Uz4OnbRRt0R3WdcPsn98zWHOaUIjWn+IfK8u6rqswywArshbs0eCauNi0/mLYP4L9D4Fo5f7gZFVDlz1IWqyRrkeH8OjsSSxoUv6MTNiVyHvqf1kCs6SakzBqPYArfQRLtCySSqAn88P5Mlj01HGLEHg6LlABaHXFNYFrdIPeE/RZgvqNM77j6iwB8Q8Qwq4OPsW6UMqHiAX4BI40DEo/i8TAEBCGoMnFEReAsief38iHzdWMsIj9r4sVGMtGif1RAmVwhLwwRC2qUW1ZYSERF4A4UilnbJYjsHbAu4dWM3Zz2XuM9C0ufADu6edtHvwbsX+StoiyUHIK+xFAi4SBahVpA1jBWbRstef37xiv3gzRIVCUd4DwGo4wLueU4bHZp/BY2HACcaJnzTBdi3cfeBefdjlwGFQr8+jG7zYOoSE0RjtH7JjSmuzvm49fdPGuxtoXH3TN9KyMsrGqFpyyzpaNqlRrn8WcHqXKDFFcU1//FJvbsusOP0Je2voyWsejVoP8H2R6+I27Fdyl62q9v9jt8HwxVL00HJHxc5EnrNoPJlMmN8Gq9CD3VY9bCg8dWyVXCLzvSsngQoAPoEEH7v+zlyO+diUlXVZvCbGyD8pQ3744D+2tvo3rgvkXFxWMZhojGaiCHLX7NRNMn5lI+dt0VRcI3C4rXErEAhfl3fgfTWF5vXl54sHJfc0dNl8D268BTvxECtCDoif8IlXmGvpCuvaPYWTMYvI9e4lGrPp6KZF+iaGPSj5tVxUaGRp2K2fNi6nXDYqB5LRjsAs/fx+vZB2L5CM63bbHMQ0HKTXsNrbjO0k3nBH5cPRU49wLXBCZIdWgxXdvKLiFiCNqHJhQHwAe3ku89KosDWqFQXcs2W+jtTx6Rgw0TyOviWqrsVjjDYg8kYax/kGzg/Js63c/lsvaxggGuvexywDqNsQyug7/onENMLZcd/cR22hePrQrW8gHmc3KoEmhUAoINj1Uf/oSscX+bhBqQbZ+wV8d4u/oW+bZn1V7xXtSRsbl7fu1VvQif1fF1C2ULcm9cALx2goaTyt5BrNC5XUJ09Hcnpe++NAQG8xmvVzuV5jLhI6gK2/c3zhQH9xHUhh1aZEP0diuuXhgYrAS+P2wYKT3aNrqNiCDYbBv4qA/1XqIvQzRjCTvsEJhSwJWNbB3mbHq6AFqoudttQrilTUw84EvlgjAWuioMKdqtmH9+//yZrjqu8oBCd80ARwfLpFdRq+2wpOGJ0JY6ntx/BIkqs0dAg97Bs0a4BN4WkfIHeiRFAGmdJHxHWxek8Ea9zS3i6Sa+aPkte0qC9Eh9kVvEMzupyBP1vaDArDa1PlGp48CQicro+jm5dZMJg/Hkg9jHjXAElZb227eE3cYMTVHQr32QJOSmJq4GSgtKF3lVM91LYTVZyXyBVqg4Gs9kHnirgE4fYJ3wv3JMQ3aW82UXLjgotkA9ukjtOfyBFfqStFXIQuY1JNyCMt+/jLT/7Q6ehLQ5lcSGtRGIRu50DxuvfcsN76OjtknK4fvvVnTDqsfLWoHHd2us9fnKsH2ts20QaqG0JVh245jp3E+xFo7CYYDbzUYD820GSoRVaHA+H9VM/83Szddk/HCYbY4SGyGSWs9ZzBaxm1fWVI01urX2IB8XVf2d9n6zk6BMJKh35y/xdy3xOCiBMtqtrdEA1hpG9r1Ma9qV8OlAQkAc3Rg1vQZhqKAB8PSqIba86N501hHN6FBM5zGPQejFtGF8fjZsynYTujuSYazjuU50jpuedph9IGF200AWzn21KIGzaSHFd8Jo34J4bBPL+yIW9mbMsqchtH/f5HV75zfBqzBpmv9Y0Li6Zy6NJj0KdHo2fCG25PZ46nfujpWTJ+5xjaF9XXT6EdlaKG7GiU68wvi62r5jXLAp9bx1/c5Cd/OHixgnQ5XHh+um1FvVgxZCj/z4HRa2pYBd0JBA2bHGT582y/7vx7c90adPOiBLpgdIndHKm9SjCugIwGX/KGx3cmbrO851cvRD79jbO6o/PXsfBgVVWjJ4R0ec1TQIoJ253RqYLuadfTETG7vdkOiFD780xDG1HvE05SD4nGC4O/rfhvacihL+h2APgXu2Yfxy50cbzD2TiOA5+D7uaeEqK9xeev6s0eR2R76UbCVBlZa3daWcdxWso4Hg1WcpGmIM1+SRgcH/uNH2PjMM8+Vmp96e6wQGHFPrfuLYzdjRMkISKmzO8A8M1lDrExouJInG+qcXTj21r4FsfzkCoi4G76Sa5EXVAzlHcnaVHMo2/r8DcYUFDgq1kHPwbETko6B609IeoV5mjmof84+qZQd0tyTHQC0oT0XMMgqZEJSCN83Kq8Wjflw2iiIxT+dsWdWrFaNdcrzQbIPsSwUen+kdL29NoeHegTH54f3IfBIcK9e8lA1sDpDMYjaqJuyH1wXMjfx3RTBpAwOvov12a4CHQaAAA="
  ],
  "requirements-export.txt": [
    "7402bf03a22395b619f2f67dac9db547880d5ba16ea077fd7f6bcd7a0734df79",
    "H4sIAAAAAAAC/yWMQQ6CMBBF95xiEpfGSQuyMKYLgwdgwwEQx3QinZK2oNzegsv338s/QLsm6wUq1Pp4hWQ5QtN2p8hPApKFgxdHkiDvkaY+9IngFbzLKUHT3W/wIBms68Mbi+TDYI0pUStUf1o4shdjFJZ13rzI1xiNpUK9QxzZbVbj5W/DLIkd7dEZq0JmN63bZ4V1MfE4+k92OqMqfsje0f+/AAAA"
  ],
  "provision.sh": [
    "13b545b6eb37ec6abf745e68a3d9759014e80329271c90bc9effa5175bdf91b0",
    "H4sIAAAAAAAC/41UYW/aMBD9nl9xylhZKzluUlatQ1TqBpP6YbRitNLUVsg4B7gEO7UdClN//C6BUqiYtEgI5+589+7duzj0wLAwkKscR0JlAS5yYz20O98uL7qDH72rbr/Tbbe00Up7tEJ6NcdA5J6N6W6Rp8IjsKenjUlp50WWAVsCY9qw9TuzKM1shjp1MCxUljJ0DrVXIgM5E1MEKvAoWOWDsfKQT8dMGj1SY8jU8DHHMUtxXp5zvTrmSz8x+mTnPEc9BymYROvVSEmC50AWNgs+wPeb9gXEJ5Bakzu4Fk6KLIIbh+AnCE7McBUyE4/GgtEwNH5CySxhdqYKkkW72+VXvT4xph08GzuNAlmkYmCN8a1gRBel0KmqeFEaap9yS8yNoP7R3es68MJZnhmqzMtrLE6iI3ih9EQ6u7WH7wOakJoA6FEjuLsDtoCwtinAh0pzPZcyhIcHODj4h48RK9YpaugFxhZzYJ0nqFvMUFDvcXIf1Ztld7oqVD5vHW1lDDfuoUUxrd5GKkiNxmCN7k+J4PVuCWorbwQcveTGsXXlyipLCGHtttP7dXnVHVy2Q2JtUyhJouPGIaTKeWtaxbDQvkiS4wY0m28xjT0xjd2Yo0NAOTEQXlszVymCWIshARIPZuCNyaYkOuKodt3r9Pu/B92Ln50Qzg+SJuCCXPFrQnRCVocpLmm24xb3s5ybHLWcf45ZNda1K0pxuGqTFEhzKFeM/srx+nIejLjwdgkJhBPvc/eV8woPJbNRap51ZkQaaYKsRET7w+mXFzTcsginURrHa6vO+eLL6eC0wbfLD+IoZvGA1q8EQkIgBmpr32qaKW0ZMPXevGe/t83/seMVjBKtytCSytnZykTra4VV6MqtrezBruD2LMhZQDLbEtY+aQe7a7Yjw3Mg3ujrpf3blLI1U2VE8BeRQfzfBwUAAA=="
  ]
}
for name, (expected, encoded) in bundle.items():
    source = gzip.decompress(base64.b64decode(encoded))
    assert hashlib.sha256(source).hexdigest() == expected, name
    (SRC / name).write_bytes(source)
    print(name, len(source), 'bytes')

## Build the CUDA-enabled Python binding

The ordinary OpenCV wheel runs CPU inference. Here CMake builds the native DNN engine for the L4's compute capability and both runtimes use the same cuDNN libraries. Compilation usually occupies most of the session. The full command logs are available under the timestamped results directory in Colab's Files pane.

In [ ]:
run(['bash', str(SRC / 'provision.sh')], 'system-setup')
# Use an interpreter whose headers and ABI are suitable for the OpenCV Python binding.
py = None
for exe in (sys.executable, shutil.which('python3.12'), shutil.which('python3.11')):
    if exe and subprocess.run([exe, '-c',
        'import sys, sysconfig, pathlib, ensurepip; assert (3,11)<=sys.version_info[:2]<=(3,12); '
        'assert pathlib.Path(sysconfig.get_path("include"),"Python.h").is_file()'],
        capture_output=True).returncode == 0:
        py = exe
        break
if not py:
    run([sys.executable, '-m', 'pip', 'install', '--target', str(ROOT / 'uv-tool'),
         'uv==0.10.2'], 'uv-install')
    uv = str(ROOT / 'uv-tool/bin/uv')
    run([uv, 'python', 'install', '3.12'], 'python-install')
    py = output([uv, 'python', 'find', '--managed-python', '3.12'])
print('Build Python', py, output([py, '--version']))

# build_cuda.sh calls python3. Put this interpreter first without changing Colab's kernel.
bin_dir = ROOT / 'tools'
bin_dir.mkdir(exist_ok=True)
shim = bin_dir / 'python3'
shim.unlink(missing_ok=True)
shim.symlink_to(py)
mem_gib = float(Path('/proc/meminfo').read_text().split('MemAvailable:')[1].split()[0]) / 1024**2
cpus = len(os.sched_getaffinity(0))
quota_path = Path('/sys/fs/cgroup/cpu.max')
if quota_path.is_file():
    quota, period = quota_path.read_text().split()
    if quota != 'max':
        cpus = min(cpus, max(1, math.floor(int(quota) / int(period))))
ram_limit = Path('/sys/fs/cgroup/memory.max')
if ram_limit.is_file() and ram_limit.read_text().strip().isdigit():
    mem_gib = min(mem_gib, int(ram_limit.read_text()) / 1024**3)
jobs = max(1, min(cpus, int(mem_gib / 2.5), 12))
threads = max(1, min(cpus, 6))
TOOLKIT = Path((ROOT / 'cuda-root').read_text().strip())
ENV = dict(os.environ, BUILD_ROOT=str(BUILD), CUDA_ROOT=str(TOOLKIT),
           CUDA_ARCH=cc.replace('.', ''), BUILD_JOBS=str(jobs),
           PATH=str(bin_dir) + os.pathsep + os.environ['PATH'])
# An inherited global override would obscure the difference between our two settings.
ENV.pop('NVIDIA_TF32_OVERRIDE', None)
print('CUDA toolkit', TOOLKIT, 'build jobs', jobs, 'inference threads', threads, flush=True)
run(['bash', str(SRC / 'build_cuda.sh')], 'opencv-build', ENV)

PYTHON = BUILD / 'venv/bin/python'
site = Path(output([str(PYTHON), '-c', 'import sysconfig;print(sysconfig.get_path("purelib"))']))
ENV['LD_LIBRARY_PATH'] = ':'.join([str(site / 'nvidia/cudnn/lib'), str(TOOLKIT / 'lib64'),
                                   ENV.get('LD_LIBRARY_PATH', '')])
run([str(PYTHON), '-m', 'unittest', '-v', 'test_benchmark'], 'regressions', ENV)
print('OpenCV build and regression checks passed.')

## Export and check the two fixed-batch models

The export environment uses CPU PyTorch and torchvision. Each export is checked against eager-model logits before benchmarking. The manifest below compares the final local files to the GTX 1080 Ti experiment, including model weights as serialized by the exporter. Within this notebook every backend reads the same local bytes.

If you have the ResNet-18 export used for the GTX 1080 Ti measurements, upload its model and export report as `/content/reference-assets.zip` before running this cell. The notebook verifies their checksums and the other local inputs, then uses the reference file for all sessions. The companion README describes the ZIP layout.

In [ ]:
REFERENCE_SHA256 = {
    "resnet18-fixed.onnx": "277c228ff616c68714a7eb838f0ef92048aa8b3b15e4e19d598782705ba01a6f",
    "vit_b_16-fixed.onnx": "2c667cf51691279a491dca8a373361bd94f0c4761eaf8188f3928a26c9e8e7f9",
    "labels.txt": "ee4fa67b1dd46919ef87529b70e25bdd1ec9ebd350de3ad426e6640eb70320a1",
    "chelsea.png": "596aa1e7cb875eb79f437e310381d26b338a81c2da23439704a73c4651e8c4bb",
    "coffee.png": "cc02f8ca188b167c775a7101b5d767d1e71792cf762c33d6fa15a4599b5a8de7"
}

# A separate CPU environment keeps the PyTorch wheels out of both timed GPU runtimes.
EXPORTER = ROOT / 'export-env'
run([py, '-m', 'venv', str(EXPORTER)], 'export-venv', ENV)
EPY = str(EXPORTER / 'bin/python')
run([EPY, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu',
     'torch==2.10.0', 'torchvision==0.25.0'], 'torch-cpu', ENV)
run([EPY, '-m', 'pip', 'install', '-r', str(SRC / 'requirements-export.txt')], 'export-deps', ENV)
run([EPY, str(SRC / 'prepare_assets.py'), '--output-dir', str(ASSETS),
     '--fixed-only'], 'export', ENV)

# Compare serialized inputs with the reference files used for the GTX 1080 Ti measurements. A local exporter
# may produce different ONNX bytes; retain that fact along with its eager-model checks.
def sha(path):
    with path.open('rb') as stream:
        digest = hashlib.sha256()
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
        return digest.hexdigest()

from zipfile import ZipFile

def use_reference_resnet(local, destination, evidence, archive):
    """Select the exact reference ResNet while preserving the original local exports."""
    model, report = 'resnet18-fixed.onnx', 'resnet18-export.json'
    expected_report = '7b988d83f44235a5438dc081a1207a04dadddcf8da33aef0a972736a12793894'
    expected_weight = 'f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec'
    with ZipFile(archive) as ref:
        if sorted(ref.namelist()) != sorted((model, report)):
            raise ValueError('reference-assets.zip must contain the reference ResNet model and its export JSON at the ZIP root.')
        info = ref.read(report)
        metadata = json.loads(info)
        assert hashlib.sha256(info).hexdigest() == expected_report, 'Unexpected reference export report'
        assert metadata['name'] == 'resnet18' and metadata['weights_sha256'] == expected_weight
        assert metadata['opset'] == 17 and metadata['exports'][model]['sha256'] == REFERENCE_SHA256[model]
        # Check the other fixed model, images and labels before reusing the local files.
        for name, expected in REFERENCE_SHA256.items():
            if name != model and sha(local / name) != expected:
                raise ValueError(f'Local {name} differs from the reference file')
        destination.mkdir(exist_ok=False)
        with ref.open(model) as src, (destination / model).open('xb') as dst:
            shutil.copyfileobj(src, dst)
        assert sha(destination / model) == REFERENCE_SHA256[model], 'Unexpected reference ResNet bytes'
        (destination / report).write_bytes(info)
    for name in ('vit_b_16-fixed.onnx', 'vit_b_16-export.json', 'labels.txt', 'chelsea.png', 'coffee.png'):
        os.link(local / name, destination / name)
    evidence.write_text(json.dumps({'imported_model': model, 'input_archive_sha256': sha(archive),
        'local_onnx_sha256': sha(local / model), 'selected_onnx_sha256': REFERENCE_SHA256[model],
        'local_export_report_sha256': sha(local / report), 'selected_export_report_sha256': expected_report},
        indent=2) + '\n')
    return destination

reference_zip = Path('/content/reference-assets.zip')
if reference_zip.is_file():
    ASSETS = use_reference_resnet(ASSETS, ROOT / f'assets-reference-{STAMP}',
                                  OUT / 'reference-import.json', reference_zip)

hashes = {name: {'actual': sha(ASSETS / name), 'reference_sha256': digest}
          for name, digest in REFERENCE_SHA256.items()}
for fname, record in hashes.items():
    record['identical'] = record['actual'] == record['reference_sha256']
    print(fname, 'identical bytes' if record['identical'] else 'different bytes', record['actual'])
    if not record['identical'] and not fname.endswith('.onnx'):
        raise ValueError(f'Input fixture mismatch: {fname}')
(OUT / 'hashes.json').write_text(json.dumps(hashes, indent=2))
for report in ASSETS.glob('*-export.json'):
    shutil.copy2(report, OUT / report.name)
env_report = {'gpu': gpu, 'cpu': output(['lscpu']), 'platform': output(['uname','-a']),
              'toolkit': output([str(TOOLKIT / 'bin/nvcc'), '--version']),
              'cv_threads': threads, 'build_jobs': jobs, 'build_python': output([py, '--version']),
              'runtime_packages': output([str(PYTHON), '-m','pip','freeze']),
              'export_packages': output([EPY, '-m','pip','freeze']),
              'opencv_build': subprocess.check_output([str(PYTHON), '-c',
                'import cv2;print(cv2.getBuildInformation())'], env=ENV, text=True)}
(OUT / 'environment.json').write_text(json.dumps(env_report, indent=2))

## Time paired sessions

Each fresh session runs 10 warm-up forwards, 50 timed forwards with host tensors, and 50 timed complete image requests including decode, resize, normalization and ranking. Each GPU case also runs in a separate untimed process to record operator placement. The two photos exercise prediction agreement. They do not measure ImageNet accuracy.

In [ ]:
# Three independent pairs per model and backend. Reverse the precision order for
# the middle pair, and the backend/model order too. Every child has a fresh session.
INDEX = []
models = ('vit_b_16', 'resnet18')
backends = ('opencv-cuda', 'ort-cuda')

def measure(model, backend, mode, trial):
    case = f'{model}_{backend}_{mode}_{trial}'
    dest = OUT / case
    cmd = [str(PYTHON), str(SRC / 'benchmark.py'), '--model', str(ASSETS / f'{model}-fixed.onnx'),
           '--image', str(ASSETS / 'chelsea.png'), '--image', str(ASSETS / 'coffee.png'),
           '--labels', str(ASSETS / 'labels.txt'), '--output', str(dest), '--backend', backend,
           '--tf32', mode, '--warmup', '10', '--runs', '50', '--threads', str(threads)]
    print(case, flush=True)
    run(cmd, case, ENV)
    result = json.loads((dest / 'result.json').read_text())
    assert result['model_sha256'] == hashes[f'{model}-fixed.onnx']['actual']
    INDEX.append({'case': case, 'model': model, 'backend': backend, 'tf32': mode, 'trial': trial})
    (OUT / 'index.json').write_text(json.dumps(INDEX, indent=2))

for model in models:
    measure(model, 'opencv-cpu', 'off', 0)
for trial in (1, 2, 3):
    order = 1 if trial % 2 else -1
    for model in models[::order]:
        for backend in backends[::order]:
            for mode in ('off', 'on')[::order]:
                measure(model, backend, mode, trial)
print(len(INDEX), 'verified sessions')

## Compare latency and logits

The table in `paired.csv` preserves each off/on pair. Compare the three pairs before interpreting a small timing difference. All full logits and individual timing samples remain available in the archive.

In [ ]:
import csv
import numpy as np
from statistics import median

def load(case):
    dest = OUT / case
    result = json.loads((dest / 'result.json').read_text())
    with np.load(dest / 'logits.npz') as logits:
        by_photo = {photo: logits[p['key']][i].copy() for p in result['predictions']
                    for i, photo in enumerate(p['images'])}
    return result, by_photo

def agreement(reference, candidate):
    assert reference.keys() == candidate.keys()
    compared = []
    for photo in sorted(reference):
        a, b = reference[photo], candidate[photo]
        assert a.shape == b.shape == (1000,) and np.isfinite(b).all()
        compared.append((int(np.argmax(a) == np.argmax(b)), float(np.max(np.abs(a - b)))))
    return sum(pair[0] for pair in compared), max(pair[1] for pair in compared)

rows = []
for model in models:
    _, cpu = load(f'{model}_opencv-cpu_off_0')
    for backend in backends:
        for trial in (1, 2, 3):
            off, logits0 = load(f'{model}_{backend}_off_{trial}')
            on, logits1 = load(f'{model}_{backend}_on_{trial}')
            count, error = agreement(logits0, logits1)
            cpu_count, cpu_error = agreement(cpu, logits1)
            rows.append({'model': model, 'backend': backend, 'trial': trial,
                         'off_forward_ms': off['forward']['p50_ms'], 'on_forward_ms': on['forward']['p50_ms'],
                         'forward_ratio_off_over_on': off['forward']['p50_ms'] / on['forward']['p50_ms'],
                         'off_app_ms': off['end_to_end']['p50_ms'], 'on_app_ms': on['end_to_end']['p50_ms'],
                         'top1_matches_on_vs_off_out_of_2': count, 'max_abs_logit_on_vs_off': error,
                         'top1_matches_on_vs_cpu_out_of_2': cpu_count, 'max_abs_logit_on_vs_cpu': cpu_error})
        paired = [r for r in rows if r['model'] == model and r['backend'] == backend]
        print(f"{model} {backend}: forward p50 (median of three sessions) "
              f"{median(r['off_forward_ms'] for r in paired):.2f} → "
              f"{median(r['on_forward_ms'] for r in paired):.2f} ms, "
              f"max |Δ logit| {max(r['max_abs_logit_on_vs_off'] for r in paired):.4g}, "
              f"on/off top-1 matches {[r['top1_matches_on_vs_off_out_of_2'] for r in paired]} / 2")
with (OUT / 'paired.csv').open('w', newline='') as stream:
    w = csv.DictWriter(stream, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

## Download the evidence

The ZIP contains each completed case and its logs. This final cell can also be run after a failure to retrieve partial work. Confirm that the file reached your computer before deleting the runtime.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

archive = ROOT / f'opencv51-l4-{STAMP}.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            z.write(path, path.relative_to(OUT))
    for path in sorted(SRC.iterdir()):
        if path.is_file():
            z.write(path, 'source/' + path.name)
with ZipFile(archive) as z:
    assert z.testzip() is None
print(archive, f'{archive.stat().st_size / (1024**2):.2f} MiB', 'SHA256', hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))